
# Amplitude-Encoded Quantum Genetic Algorithm (AEQGA)
## Cosmological Parameter Estimation with Sarracino et al.'s Algorithm

**Paper**: Sarracino et al., *"A Quantum Genetic Algorithm with application to Cosmological Parameters Estimation"*, Astron. Comput. 55:101078 (2026), arXiv:[2602.15459](https://arxiv.org/abs/2602.15459)

---

This notebook reproduces the full AEQGA pipeline described in §3 of the paper:

1. **Amplitude encoding** of a cosmological parameter population (§3.1, Eq.11)
2. **Quantum crossover** via coupled controlled-$R_y$ rotations (§3.2, Eq.12-14)
3. **Quantum mutation** via single-qubit $R_x$ rotations (§3.2, Eq.15)
4. **Quantum decoding** from measurement counts (§3.3, Eq.16-18)
5. **Classical fitness evaluation** on $\chi^2$ (§2)
6. **Iteration** over generations to find best-fit $(H_0, \Omega_M)$

The algorithm searches for the best-fit cosmological parameters that minimise the Pantheon SNe Ia $\chi^2$ likelihood.

### Prerequisites

```bash
pip install qiskit qiskit-aer tqdm numpy matplotlib
```

### Data

Clone the Pantheon SNe Ia dataset:

```bash
git clone https://github.com/CobayaSampler/sn_data
```

The expected directory structure: `sn_data/Pantheon/lcparam_full_long_zhel.txt`


In [1]:

# =============================================================================
# IMPORTS
# =============================================================================

import math
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Qiskit imports
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Local modules
import amplitude_encoding
import quantum_gates
import aeqga_algorithm
import pantheon_problem

print("All imports successful.")
import qiskit
print(f"Qiskit version: {qiskit.__version__}")
print(f"NumPy version:  {np.__version__}")


All imports successful.
Qiskit version: 2.5.2
NumPy version:  2.5.3



---

## 1. Amplitude Encoding (§3.1, Eq.11)

### Theory

The paper's amplitude encoding maps a classical population vector

$$\mathbf{x} = [x_0, x_1, \ldots, x_{N-1}]$$

onto a quantum state by first normalising it so that $\sum_i |x_i|^2 = 1$, then assigning each value as the **amplitude** of a computational basis state:

$$|\psi\rangle = \sum_{i=0}^{N-1} x_i |i\rangle$$

where $|i\rangle$ is the $i$-th computational basis state of an $n = \log_2 N$ qubit system.

**Key properties**:

- The number of qubits scales **logarithmically** with population size: $n = \log_2 N$.
- For a population of $n_p = 32$ individuals, we need only $n = 5$ qubits.
- The encoding is implemented via `Qiskit.initialize()`, which decomposes the state preparation into elementary rotation gates (see Fig. 1 of the paper).
- The algorithm uses **two circuits per parameter**: one for the elite-copy subset ($n_q = \log_2 n_p - 2$ qubits) and one for the random subset ($n_q = \log_2 n_p - 1$ qubits).

### Implementation

The `amplitude_encoding.py` module provides:

- `_normalise(x)`: L2-normalises a vector (handles zero-norm edge case)
- `n_qubits_for_population(n_p)`: Returns $\log_2 n_p$; asserts power-of-two
- `build_amplitude_circuit(values)`: Creates the quantum circuit with `initialize`


In [2]:

# =============================================================================
# STEP 1: DEMONSTRATE AMPLITUDE ENCODING
# =============================================================================

print("=" * 60)
print("  STEP 1: Amplitude Encoding Demo")
print("=" * 60)

# Example: population of 8 individuals (n_p = 2^3)
n_p = 8
values = np.array([0.1, 0.3, 0.5, 0.7, 0.2, 0.4, 0.6, 0.8])

# Verify qubit count
n_qubits = amplitude_encoding.n_qubits_for_population(n_p)
print(f"\nPopulation size: {n_p} individuals")
print(f"Qubits needed:   n = log2({n_p}) = {n_qubits}")
assert 2 ** n_qubits == n_p

# Normalise and build circuit
norm = amplitude_encoding._normalise(values)
print(f"\nOriginal values:      {values}")
print(f"L2-normalised:        {norm}")
print(f"Sum of squares:       {np.sum(norm**2):.10f}  (should be 1.0)")

# Build the quantum circuit
qc = amplitude_encoding.build_amplitude_circuit(values)
print(f"\nQuantum circuit: {qc.num_qubits} qubits, {qc.size()} gates")
print(qc.draw(output='text', fold=-1))

# Verify the statevector
from qiskit.quantum_info import Statevector
sv = Statevector(qc)
print(f"\nStatevector (first 4 amplitudes): {sv.data[:4]}")
print(f"Sum of |amplitude|^2: {np.sum(np.abs(sv.data)**2):.10f}")

# Power-of-two enforcement
try:
    amplitude_encoding.n_qubits_for_population(10)
    print("\nERROR: Should have raised AssertionError!")
except AssertionError as e:
    print(f"\nNon-power-of-two correctly rejected: {e}")

print("\n✓ Amplitude encoding verified.")


  STEP 1: Amplitude Encoding Demo

Population size: 8 individuals
Qubits needed:   n = log2(8) = 3

Original values:      [0.1 0.3 0.5 0.7 0.2 0.4 0.6 0.8]
L2-normalised:        [0.070014   0.21004201 0.35007002 0.49009803 0.14002801 0.28005602
 0.42008403 0.56011203]
Sum of squares:       1.0000000000  (should be 1.0)

Quantum circuit: 3 qubits, 1 gates
     ┌──────────────────────────────────────────────────────────────────────────────┐
q_0: ┤0                                                                             ├
     │                                                                              │
q_1: ┤1 Initialize(0.070014,0.21004,0.35007,0.4901,0.14003,0.28006,0.42008,0.56011) ├
     │                                                                              │
q_2: ┤2                                                                             ├
     └──────────────────────────────────────────────────────────────────────────────┘

Statevector (first 4 amplitudes): [0.070


---

## 2. Quantum Gates (§3.2, Eq.12-15)

### Quantum Crossover (§3.2, Eq.12-14)

The crossover operation entangles two qubits using **coupled controlled-$R_y$** rotations. The paper specifies a fixed rotation angle of $\pi/2$, chosen to maximise the variation in probability distributions:

$$\text{CR}_y\left(\frac{\pi}{2}\right) = |0\rangle\langle 0| \otimes I + |1\rangle\langle 1| \otimes R_y\left(\frac{\pi}{2}\right)$$

where:

$$R_y\left(\frac{\pi}{2}\right) = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & -1 \\ 1 & 1 \end{pmatrix}$$

The total crossover unitary is the product applied **bidirectionally**:

$$U_{\text{cross}} = \text{CR}_{y,0\to1}\left(\frac{\pi}{2}\right) \cdot \text{CR}_{y,1\to0}\left(\frac{\pi}{2}\right)$$

$$U_{\text{cross}} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & \frac{1}{\sqrt{2}} & -\frac{1}{2} & -\frac{1}{2} \\ 0 & 0 & \frac{1}{\sqrt{2}} & -\frac{1}{\sqrt{2}} \\ 0 & \frac{1}{\sqrt{2}} & \frac{1}{2} & \frac{1}{2} \end{pmatrix}$$

### Quantum Mutation (§3.2, Eq.15)

Mutation applies a single-qubit $R_x$ rotation of fixed angle $\pi/2$ on a randomly selected qubit:

$$U_{\text{mut}} = R_x\left(\frac{\pi}{2}\right) = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & -i \\ -i & 1 \end{pmatrix}$$

Note the intentional avoidance of the $z$-axis: rotations around $z$ produce only phase shifts, which do not affect measurement probabilities.

### Implementation

The `quantum_gates.py` module provides:

- `apply_crossover(qc, p_c)`: Applies $\text{CR}_y(\pi/2)$ bidirectionally with probability $p_c$
- `apply_mutation(qc, p_m)`: Applies $R_x(\pi/2)$ to a random qubit with probability $p_m$


In [3]:

# =============================================================================
# STEP 2: VERIFY QUANTUM GATES
# =============================================================================

print("=" * 60)
print("  STEP 2: Quantum Gate Verification")
print("=" * 60)

from qiskit.quantum_info import Operator
import quantum_gates

# Verify U_cross (Eq.14)
print("\n--- Crossover Verification ---")
mat, U_cross_expected = quantum_gates.verify_u_cross()
print(f"CRy(π/2) bidirectional matches Eq.14: {np.allclose(mat, U_cross_expected, atol=1e-10)}")
print(f"\nU_cross matrix:\n{np.real_if_close(mat, tol=1e-10)}")

# Verify U_mut (Eq.15)
print("\n--- Mutation Verification ---")
rx_mat = quantum_gates.verify_rx_pi2()
U_mut_expected = (1/np.sqrt(2)) * np.array([[1, -1j], [-1j, 1]], dtype=complex)
print(f"Rx(π/2) matches Eq.15: {np.allclose(rx_mat, U_mut_expected, atol=1e-10)}")
print(f"\nRx(π/2) matrix:\n{np.real_if_close(rx_mat, tol=1e-10)}")

print("\n✓ All quantum gates verified against paper equations.")


  STEP 2: Quantum Gate Verification

--- Crossover Verification ---
CRy(π/2) bidirectional matches Eq.14: True

U_cross matrix:
[[ 1.          0.          0.          0.        ]
 [ 0.          0.70710678 -0.5        -0.5       ]
 [ 0.          0.          0.70710678 -0.70710678]
 [ 0.          0.70710678  0.5         0.5       ]]

--- Mutation Verification ---
Rx(π/2) matches Eq.15: True

Rx(π/2) matrix:
[[0.70710678+0.j         0.        -0.70710678j]
 [0.        -0.70710678j 0.70710678+0.j        ]]

✓ All quantum gates verified against paper equations.



---

## 3. Quantum Decoding (§3.3, Eq.16-18)

### Theory

After measurement, the algorithm obtains **counts per computational basis state** $c_i$ (not per-qubit marginals). The paper uses two different decoding schemes for two sub-populations:

#### Random Subset (Eq.16)

For the randomly drawn population, decode via **full-range min-max scaling**:

$$x_i = a + (b - a) \cdot \frac{c_i - c_{\min}}{c_{\max} - c_{\min}}$$

where $[a, b]$ is the parameter range (e.g., $[0.0, 0.5]$ for $\Omega_M$).

**Post-processing**: if $x_i = a$ exactly (i.e., $c_i = c_{\min}$ for multiple states), replace with a draw from $\mathcal{U}(a, b)$ to avoid collapse to the lower bound.

#### Elite-Copy Subset (Eq.17-18)

For the elite-copy population, decode within a **restricted box** around the previous generation's best individuals:

$$p_i = \frac{c_i}{\sum_j c_j}$$

$$x_i = n_{\min} + (n_{\max} - n_{\min}) \cdot \sqrt{p_i}$$

where $\sqrt{p_i}$ phenomenologically focuses samples toward the box center. The box $[n_{\min}, n_{\max}]$ is computed around the elite span (expanded by 5%).

### Implementation

The `amplitude_encoding.py` module provides:

- `decode_random_subset(counts, n_qubits, a, b, n)`: Eq.16 decoding
- `decode_elite_subset(counts, n_qubits, n_min, n_max, n)`: Eq.17-18 decoding


In [4]:

# =============================================================================
# STEP 3: DEMONSTRATE DECODING
# =============================================================================

print("=" * 60)
print("  STEP 3: Decoding Demo")
print("=" * 60)

# Simulate measurement counts for 8 individuals in 3 qubits (n_p=8)
n_qubits = 3
n_states = 2 ** n_qubits
np.random.seed(42)

# Random counts (simulating shot-based measurement)
counts_rand = {format(i, f'0{n_qubits}b'): max(1, int(np.random.exponential(50))) for i in range(n_states)}
print(f"\nRandom subset counts ({n_states} basis states):")
for k, v in sorted(counts_rand.items(), key=lambda x: int(x[0], 2)):
    print(f"  |{k}\rangle: {v} shots")

# Decode random subset (Eq.16)
a, b = 0.0, 0.5  # Ω_M range
decoded_rand = amplitude_encoding.decode_random_subset(counts_rand, n_qubits, a, b, 8)
print(f"\nDecoded random values ({len(decoded_rand)} individuals):")
print(f"  min={decoded_rand.min():.4f}, max={decoded_rand.max():.4f}")
print(f"  mean={decoded_rand.mean():.4f}")

# Simulate counts for elite-copy subset
counts_elite = {format(i, f'0{n_qubits}b'): max(1, int(np.random.exponential(100))) for i in range(n_states)}
n_min_e, n_max_e = 0.3, 0.4  # Box around previous elites

# Decode elite-copy subset (Eq.17-18)
decoded_elite = amplitude_encoding.decode_elite_subset(counts_elite, n_qubits, n_min_e, n_max_e, 2)
print(f"\nDecoded elite-copy values ({len(decoded_elite)} individuals, box=[{n_min_e}, {n_max_e}]):")
print(f"  values: {decoded_elite}")
print(f"  all in box: {np.all((decoded_elite >= n_min_e) & (decoded_elite <= n_max_e))}")

# Test zero-count edge case
counts_zero = {format(i, f'0{n_qubits}b'): 100 for i in range(n_states)}  # all equal
decoded = amplitude_encoding.decode_random_subset(counts_zero, n_qubits, a, b, 2)
print(f"\nEqual counts -> uniform draw: {decoded}")

print("\n✓ Decoding verified.")


  STEP 3: Decoding Demo

Random subset counts (8 basis states):
angle: 23 shots
angle: 150 shots
angle: 65 shots
angle: 45 shots
angle: 8 shots
angle: 8 shots
angle: 2 shots
angle: 100 shots

Decoded random values (8 individuals):
  min=0.0203, max=0.5000
  mean=0.2002

Decoded elite-copy values (2 individuals, box=[0.3, 0.4]):
  values: [0.34044303 0.30515711]
  all in box: True

Equal counts -> uniform draw: [0.26237822 0.21597251]

✓ Decoding verified.



---

## 4. Cosmological Data (§2)

### Pantheon SNe Ia Dataset

The paper uses the **Pantheon+** sample (Scolnic+18, Brout+18) of 1701 SNe Ia. The $\chi^2$ likelihood is:

$$\chi^2 = (\mu_{\text{th}} - \mu_{\text{obs}})^T \mathcal{C}^{-1} (\mu_{\text{th}} - \mu_{\text{obs}})$$

where:
- $\mu_{\text{th}} = 5\log_{10}(d_L(z_{\text{cmb}}, z_{\text{hel}}) / 10\,\text{pc}) + 25$ is the theoretical distance modulus
- $d_L$ is the flat $\Lambda$CDM luminosity distance: $d_L = (1+z_{\text{hel}}) \frac{c}{H_0} \int_0^{z} \frac{dz'}{\sqrt{\Omega_M(1+z')^3 + \Omega_\Lambda}}$
- $\mathcal{C}$ is the statistical + systematic covariance matrix
- The absolute magnitude $M$ is analytically marginalised (Conley+11, Eq. C1)

### Parameter Search Space (§3 p.7)

$$\Omega_M \in [0.0,\, 0.5], \qquad H_0 \in [60,\, 80]\,\text{km/s/Mpc}$$

### Data Setup

Clone the data repository:

```bash
git clone https://github.com/CobayaSampler/sn_data
```

Expected file: `sn_data/Pantheon/lcparam_full_long_zhel.txt`


In [5]:

# =============================================================================
# STEP 4: LOAD COSMOLOGICAL DATA
# =============================================================================

print("=" * 60)
print("  STEP 4: Load Pantheon Data")
print("=" * 60)

# Adjust this path to your local sn_data/Pantheon directory
DATA_DIR = "sn_data/Pantheon"

try:
    from pantheon_problem import load_pantheon, PantheonProblem
    
    data = load_pantheon(DATA_DIR)
    print(f"\nLoaded Pantheon data:")
    print(f"  Number of SNe:         {data['n_sn']}")
    print(f"  Redshift range:        [{data['zcmb'].min():.4f}, {data['zcmb'].max():.4f}]")
    print(f"  Distance modulus range: [{data['mb'].min():.2f}, {data['mb'].max():.2f}]")
    print(f"  Covariance:            {'stat+sys' if data['cov_sys'] is not None else 'stat only'}")
    
    # Build the problem instance
    problem = PantheonProblem(DATA_DIR, use_full_cov=True, verbose=False)
    print(f"\nProblem bounds:")
    print(f"  H0:       [{problem.lower_bounds[0]}, {problem.upper_bounds[0]}]")
    print(f"  Omega_M:  [{problem.lower_bounds[1]}, {problem.upper_bounds[1]}]")
    
except FileNotFoundError:
    print(f"\nData not found: sn_data/Pantheon not found.")
    print("To use this notebook, clone the data:")
    print("  git clone https://github.com/CobayaSampler/sn_data")
    print("  and set DATA_DIR = 'sn_data/Pantheon'")
    print("\nFalling back to a mock problem for demonstration...")
    
    class MockProblem:
        lower_bounds = np.array([60.0, 0.0])
        upper_bounds = np.array([80.0, 0.5])
        n_dim = 2
        def compute_fitness(self, x):
            H0, Om = float(x[0]), float(x[1])
            if Om <= 0 or Om >= 0.5 or H0 <= 0 or H0 > 80:
                return 1e12
            return (H0 - 72.82)**2 / 0.22**2 + (Om - 0.363)**2 / 0.016**2
        def is_max_problem(self):
            return False
    
    problem = MockProblem()
    print("Using mock problem with best-fit near (H0=72.82, Omega_M=0.363).")


  STEP 4: Load Pantheon Data

Data not found: sn_data/Pantheon not found.
To use this notebook, clone the data:
  git clone https://github.com/CobayaSampler/sn_data
  and set DATA_DIR = 'sn_data/Pantheon'

Falling back to a mock problem for demonstration...
Using mock problem with best-fit near (H0=72.82, Omega_M=0.363).



---

## 5. AEQGA Algorithm (§3, Alg.1)

### Overview

The algorithm proceeds as follows for each generation $t = 1, \ldots, n_g$:

```
Algorithm 1: Amplitude-Encoded Quantum Genetic Algorithm
Input: Population size n_p, generations n_g, p_c, p_m
1.  Evaluate chi^2(x) classically for every x in P
2.  Select top 25% as P_elite (bypass quantum circuits)
3.  Duplicate P_elite -> P_elite_copy (n_p/4)
4.  Draw fresh P_rand uniform [a,b] (n_p/2)
5.  For each dimension d:
      a. Encode P_rand[:,d] -> n_q = log2(n_p)-1 qubits
      b. Encode P_elite_copy[:,d] -> n_q = log2(n_p)-2 qubits
      c. Apply CRy(pi/2) crossover with prob p_c
      d. Apply Rx(pi/2) mutation with prob p_m
      e. Measure -> decode via Eq.16 (random) or Eq.17-18 (elite)
6.  Combine P <- P_elite ∪ P_decoded
7.  Repeat until n_g generations reached
```

### Key Design Decisions

| Aspect | Paper Specification | Why |
|--------|-------------------|-----|
| Population split | 25% elite / 25% duplicate / 50% random | Ensures monotonic improvement |
| Qubit count | log2(n_p) - 1 (random), log2(n_p) - 2 (elite) | Logarithmic encoding efficiency |
| Crossover angle | Fixed pi/2 | Maximises probability variation |
| Mutation angle | Fixed pi/2 | Deterministic, no hyperparameter |
| Crossover axis | y-axis | Maximally entangling |
| Mutation axis | x-axis | Changes probabilities (not phase) |
| Optimal p_c, p_m | 0.5 each | Paper §4.1 shows best precision |
| Population size | Power of two | Required for log2 qubit count |

### Implementation

The `aeqga_algorithm.py` module provides:

- `AEQGAParameters`: Hyperparameter container
- `build_dimension_circuit(values, p_cross, p_mut, num_shots, mode)`: Builds the per-dimension circuit
- `split_population(population, fitnesses, minimise, lower, upper)`: Alg.1 L5-L7 split
- `run_aeqga_dual(problem, params)`: Full algorithm loop (Alg.1)
- `run_aeqga_sv(problem, params)`: Statevector variant (no shot noise)
- `run_aeqga_iterations(problem, params)`: Outer loop for n_i=300 statistics (§3.4)


In [6]:

# =============================================================================
# STEP 5: CONFIGURE AEQGA PARAMETERS
# =============================================================================

print("=" * 60)
print("  STEP 5: Configure Parameters")
print("=" * 60)

params = aeqga_algorithm.AEQGAParameters(
    pop_size     = 8,      # MUST be power of two (2^3)
    max_gen      = 10,     # Number of generations (paper uses 50)
    n_iterations = 1,      # Outer runs for statistics (paper uses 300)
    p_cross      = 0.5,    # Paper's optimal crossover probability (§4.1)
    p_mut        = 0.5,    # Paper's optimal mutation probability (§4.1)
    num_shots    = 4096,   # Shots per circuit (0 = statevector mode)
    verbose      = False,
    progress_bar = True,
)

# Validate parameters
params._validate()
print(f"\nAEQGAParameters:")
print(f"  pop_size:      {params.pop_size}  (= 2^{int(np.log2(params.pop_size))})")
print(f"  max_gen:       {params.max_gen}")
print(f"  n_iterations:  {params.n_iterations}")
print(f"  p_cross:       {params.p_cross}")
print(f"  p_mut:         {params.p_mut}")
print(f"  num_shots:     {params.num_shots}")
print(f"  sigma_mut:     {getattr(params, 'sigma_mut', 'removed (paper has no Gaussian mutation)')}")
print(f"\nParameter validation passed.")


  STEP 5: Configure Parameters

AEQGAParameters:
  pop_size:      8  (= 2^3)
  max_gen:       10
  n_iterations:  1
  p_cross:       0.5
  p_mut:         0.5
  num_shots:     4096
  sigma_mut:     removed (paper has no Gaussian mutation)

Parameter validation passed.


In [7]:

# =============================================================================
# STEP 6: RUN THE AEQGA
# =============================================================================

print("=" * 60)
print("  STEP 6: Running AEQGA")
print("=" * 60)

# Use the quantum simulator
simulator = AerSimulator()

print(f"\nRunning AEQGA for {params.max_gen} generations with {params.pop_size} individuals...")
print(f"  Population split: 25% elite / 25% duplicate / 50% random")
print(f"  Search space: H0 in [{problem.lower_bounds[0]}, {problem.upper_bounds[0]}],")
print(f"                Omega_M in [{problem.lower_bounds[1]}, {problem.upper_bounds[1]}]")
print()

# Run the algorithm
g_best, population_evol, bests_log = aeqga_algorithm.run_aeqga_dual(
    problem, params, simulator=simulator
)

H0_best = g_best.x[0]
Om_best = g_best.x[1]
chi2_best = g_best.fitness

print(f"\n{'='*60}")
print(f"  RESULTS")
print(f"{'='*60}")
print(f"  Best-fit H0      = {H0_best:.4f}  km/s/Mpc")
print(f"  Best-fit Omega_M = {Om_best:.4f}")
print(f"  chi^2_min          = {chi2_best:.4f}")
print(f"  Found at gen     = {g_best.gen}")
print()
print(f"  Paper SNe Ia result: Omega_M = 0.363+/-0.016, H0 = 72.81+/-0.22")
print(f"  Reference (Scolnic+18): H0 ~ 67.4, Omega_m ~ 0.298")

# Store results
import json
results = {
    'H0_best': float(H0_best),
    'Om_best': float(Om_best),
    'chi2_best': float(chi2_best),
    'gen': int(g_best.gen),
    'bests_log': [[x.tolist() if hasattr(x, 'tolist') else x, float(f)] for x, f in bests_log],
}
with open('aeqga_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to aeqga_results.json")


  STEP 6: Running AEQGA

Running AEQGA for 10 generations with 8 individuals...
  Population split: 25% elite / 25% duplicate / 50% random
  Search space: H0 in [60.0, 80.0],
                Omega_M in [0.0, 0.5]



AEQGA generations: 100%|██████████| 11/11 [00:02<00:00,  4.31it/s]


[GlobalBest] gen=1  fitness=18.1282  x=[72.96956459  0.29575041]
Total merit-function evaluations: 88

  RESULTS
  Best-fit H0      = 72.9696  km/s/Mpc
  Best-fit Omega_M = 0.2958
  chi^2_min          = 18.1282
  Found at gen     = 1

  Paper SNe Ia result: Omega_M = 0.363+/-0.016, H0 = 72.81+/-0.22
  Reference (Scolnic+18): H0 ~ 67.4, Omega_m ~ 0.298


TypeError: only 0-dimensional arrays can be converted to Python scalars

In [ ]:

# =============================================================================
# STEP 7: CONVERGENCE ANALYSIS
# =============================================================================

print("=" * 60)
print("  STEP 7: Convergence Analysis")
print("=" * 60)

# Plot convergence
gens = list(range(len(bests_log)))
fitvals = [b[1] for b in bests_log]

plt.figure(figsize=(10, 5))
plt.plot(gens, fitvals, linewidth=2, color='steelblue', label='AEQGA best chi^2')
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Best $\\chi^2$', fontsize=12)
plt.title('AEQGA Convergence — Pantheon SNe Ia', fontsize=13)
plt.grid(True, alpha=0.35)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('pantheon_convergence.png', dpi=150)
plt.show()
print("\nConvergence plot saved to pantheon_convergence.png")

# Plot parameter evolution
if len(population_evol) > 2:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    h0_evo = [b[0][0] for b in bests_log]
    om_evo = [b[0][1] for b in bests_log]
    
    axes[0].plot(h0_evo, 'b-', linewidth=1.5)
    axes[0].axhline(y=72.82, color='r', linestyle='--', label='Paper best-fit H0=72.82')
    axes[0].set_xlabel('Generation')
    axes[0].set_ylabel('$H_0$ (km/s/Mpc)')
    axes[0].set_title('H0 Evolution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.35)
    
    axes[1].plot(om_evo, 'b-', linewidth=1.5)
    axes[1].axhline(y=0.363, color='r', linestyle='--', label='Paper best-fit Omega_M=0.363')
    axes[1].set_xlabel('Generation')
    axes[1].set_ylabel('$\\Omega_M$')
    axes[1].set_title('$\\Omega_M$ Evolution')
    axes[1].legend()
    axes[1].grid(True, alpha=0.35)
    
    plt.tight_layout()
    plt.savefig('aeqga_parameter_evolution.png', dpi=150)
    plt.show()
    print("Parameter evolution plot saved to aeqga_parameter_evolution.png")



---

## 6. Statistical Analysis (§3.4, n_i = 300)

The paper reports results as mean ± standard deviation over $n_i = 300$ independent runs. For SNe Ia:

$$\Omega_M = 0.362 \pm 0.016, \quad H_0 = 72.81 \pm 0.22$$

The following runs `n_iterations` independent AEQGA runs to estimate the mean and standard deviation.

**Note**: This requires `n_iterations × max_gen` circuit executions. For production results, use `n_iterations=300`, `max_gen=50`, `pop_size=32`.


In [ ]:

# =============================================================================
# STEP 8: STATISTICAL ANALYSIS
# =============================================================================

print("=" * 60)
print("  STEP 8: Statistical Analysis")
print("=" * 60)

params_iter = aeqga_algorithm.AEQGAParameters(
    pop_size=8, max_gen=5, n_iterations=3,  # Use 3 for demo; paper uses 300
    p_cross=0.5, p_mut=0.5, num_shots=4096,
    verbose=False, progress_bar=True,
)

means, stds, _ = aeqga_algorithm.run_aeqga_iterations(problem, params_iter, simulator=simulator)
print(f"\nPaper §3.4 reference: Omega_M = 0.363+/-0.016, H0 = 72.82+/-0.22")
print(f"Our results (n={params_iter.n_iterations} iters): Omega_M = {means[1]:.4f}+/-{stds[1]:.4f}, H0 = {means[0]:.4f}+/-{stds[0]:.4f}")

# For production: use n_iterations=300
# params_iter.n_iterations = 300
# means, stds, _ = aeqga_algorithm.run_aeqga_iterations(problem, params_iter, simulator)



---

## 7. Statevector Variant (Shot-Noise-Free)

The paper's "probabilities" path uses exact statevector probabilities rather than shot-sampled counts, removing shot noise. The `run_aeqga_sv` function uses `AerSimulator(method="statevector")`.

Note: For $n_p=8$, the statevector has $2^3=8$ amplitudes per dimension (trivial). For $n_p=1024$, it would need $2^{10}=1024$ amplitudes.


In [ ]:

# =============================================================================
# STEP 9: STATEVECTOR VARIANT
# =============================================================================

print("=" * 60)
print("  STEP 9: Statevector Variant")
print("=" * 60)

params_sv = aeqga_algorithm.AEQGAParameters(
    pop_size=8, max_gen=5, p_cross=0.5, p_mut=0.5, num_shots=0,
    verbose=False, progress_bar=True,
)

g_best_sv, _, bests_log_sv = aeqga_algorithm.run_aeqga_sv(problem, params_sv, simulator=simulator)
print(f"\nStatevector results: H0={g_best_sv.x[0]:.4f}, Omega_M={g_best_sv.x[1]:.4f}")
print("✓ Statevector variant complete (no shot noise).")



---

## 8. Extending to BAO and CMB (§2.2-2.3)

### BAO Data (§2.2, Eq.6-10)

The paper also minimises the BAO $\chi^2$ using 16 measurements. The acoustic scale $r_d$ (Eq.9) uses fixed parameters:

$$r_d = \frac{55.154 \cdot e^{-72.3(\Omega_\nu h^2 + 0.0006)^2}}{(\Omega_M h^2)^{0.25351}(\Omega_b h^2)^{0.12807}} \text{ Mpc}$$

with $\Omega_b h^2 = 0.02237$, $\Omega_\nu h^2 = 0.00064$, $\Sigma m_\nu \approx 0.06$ eV.

### CMB Data (§2.3)

The Planck TT power spectrum is compared using CAMB or the PICO emulator.

### Current Status

| Component | Status | File |
|-----------|--------|------|
| SNe Ia | ✅ Full | `pantheon_problem.py` |
| BAO | ⏳ Stub | `chi2_bao()` returns 0.0 |
| CMB | ⏳ Stub | `chi2_cmb()` returns 0.0 |
| Combined | ⏳ Future | Sum of $\chi^2_{SNe} + \chi^2_{BAO} + \chi^2_{CMB}$ |



---

## References

1. **Sarracino et al.** (2026). *"A Quantum Genetic Algorithm with application to Cosmological Parameters Estimation"*. Astron. Comput. 55:101078. arXiv:[2602.15459](https://arxiv.org/abs/2602.15459).

2. **Scolnic et al.** (2018). *"The Complete Type Ia Supernova Sample from the Nearby Universe"*. ApJ 859:101.

3. **Conley et al.** (2011). *"The Supernova Mass Analyzer"*. ApJ 740:12.

4. **Fendt & Wandelt** (2007). *"A Fast Approximation to Cosmic Microwave Background Data"*. Phys. Rev. D 75:083007. (PICO emulator)

5. **Qiskit Documentation**: [https://quantum.cloud.ibm.com/docs](https://quantum.cloud.ibm.com/docs)

---

*Notebook implementing the AEQGA algorithm as described in Sarracino et al. (2026), based on the AEGQA-main workspace.*
